# Tuning `motion_detection` interactively — live video feed

A calibration notebook: pick **any** video, then drag sliders to find the detection
parameters that work for it. On every slider **release**, detection re-runs (via the
`MotionDetectionOptions.from_gui` composer) over a short **preview window** and the notebook
regenerates a short clip that plays inline:

* **left** — the video frame with the cell grid, numbered badges and a highlight on active cells;
* **right** — the temporal-std **heatmap** (same grid);
* **bottom** — the per-quadrant energy timeline with the threshold, the shaded active spans,
  and a cursor tracking the current frame.

The clip is a real, scrubbable, playable `<video>` — that is your "video feed".

## Why it regenerates on release (not while dragging)

Re-running detection *and* encoding video for every pixel of a drag is too slow, so the
parameter sliders use `continuous_update=False`: detection + encoding happen when you let go.
Lower **Preview sec** if regeneration feels sluggish on a long / high-resolution clip.

In [ ]:
import os
import subprocess
import shutil
import tempfile
import base64
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["figure.dpi"] = 110

import ipywidgets as widgets
from IPython.display import display, Video, HTML

from saltup.utils.data.video.video_utils import (
    motion_detection,
    process_video,
    get_video_properties,
    VideoReadOptions,
    MotionDetectionOptions,
    _compute_grid,
    _quadrant_names,
)
from saltup.utils.data.image.image_utils import Image

## Helpers

Reused from the library's own grid geometry so the visualisation and the detection always
agree. `collect_visuals` rebuilds the frames + heatmap with the **same** preprocessing
(reusing `process_video` + `Image`); `make_player_from_data` encodes the precomputed frames,
heatmap and energy signal into one short clip.

In [ ]:
def cell_slices(H, W, rows, cols):
    ys = np.linspace(0, H, rows + 1).round().astype(int)
    xs = np.linspace(0, W, cols + 1).round().astype(int)
    return [(ys[r], ys[r + 1], xs[c], xs[c + 1]) for r in range(rows) for c in range(cols)]


def make_grid_visual(n_quadrants):
    rows, cols = _compute_grid(n_quadrants)
    names = _quadrant_names(n_quadrants)
    colors = [plt.cm.tab10(i % 10) for i in range(n_quadrants)]
    return rows, cols, names, colors


def _runs(mask):
    runs, s = [], None
    for i, m in enumerate(mask):
        if m and s is None:
            s = i
        elif not m and s is not None:
            runs.append((s, i - 1))
            s = None
    if s is not None:
        runs.append((s, len(mask) - 1))
    return runs


def draw_grid(panel, h, w, rows, cols, colors, active_row):
    ys = np.linspace(0, h, rows + 1).round().astype(int)
    xs = np.linspace(0, w, cols + 1).round().astype(int)
    for y in ys[1:-1]:
        cv2.line(panel, (0, y), (w, y), (255, 255, 255), 1)
    for x in xs[1:-1]:
        cv2.line(panel, (x, 0), (x, h), (255, 255, 255), 1)
    for qi, (y0, y1, x0, x1) in enumerate(cell_slices(h, w, rows, cols)):
        if active_row[qi]:
            cv2.rectangle(panel, (x0 + 1, y0 + 1), (x1 - 2, y1 - 2), colors[qi], 2)


def collect_visuals(path, config, fps, start_s=0.0, duration_seconds=None):
    """Rebuild frames + temporal-std heatmap with the same preprocessing as motion_detection."""
    nq = config.n_quadrants
    r, c = _compute_grid(nq)
    rw = config.resize_width
    N = max(2, int(round(config.window_seconds * fps)))
    alpha = 1.0 / N
    frames_rgb, std_maps = [], []
    state = {"mean": None, "sq": None}

    def cb(img, frame_number, total_frames, meta):
        arr = img.get_data()
        h0, w0 = arr.shape[:2]
        h = int(round(h0 * rw / w0))
        small = cv2.resize(arr, (rw, h), interpolation=cv2.INTER_AREA)
        gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY).astype(np.float32)
        for (y0, y1, x0, x1) in cell_slices(*gray.shape, r, c):
            gray[y0:y1, x0:x1] -= gray[y0:y1, x0:x1].mean()
        if state["mean"] is None:
            state["mean"] = gray.copy()
            state["sq"] = gray * gray
        else:
            cv2.accumulateWeighted(gray, state["mean"], alpha)
            cv2.accumulateWeighted(gray * gray, state["sq"], alpha)
        std_maps.append(np.sqrt(np.clip(state["sq"] - state["mean"] ** 2, 0, None)).astype(np.float32))
        frames_rgb.append(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))
        return img

    if duration_seconds:
        fn = list(range(int(round(start_s * fps)), int(round((start_s + duration_seconds) * fps))))
        process_video(path, callback=cb, frame_numbers=fn, options=opts)
    else:
        process_video(path, callback=cb, options=opts)
    return np.asarray(frames_rgb), np.asarray(std_maps)


def show_video(path):
    """Display a local video as an autoplaying, scrubbable HTML5 clip (IPython-version agnostic)."""
    data = base64.b64encode(Path(path).read_bytes()).decode("ascii")
    display(HTML(
        f'<video src="data:video/mp4;base64,{data}" autoplay muted controls loop '
        f'style="width:100%">Your browser does not support the video tag.</video>'))


def make_player_from_data(frames_rgb, std_maps, active, signal, times, threshold,
                          rows, cols, colors, fps, heat_vmax=12.0, out_path="motion_tuning_player.mp4"):
    """Encode the precomputed frames/heatmap/energy into one short, scrubbable clip."""
    T = len(times)
    h, W = frames_rgb.shape[1], frames_rgb.shape[2]
    bgr = [(int(255 * c[2]), int(255 * c[1]), int(255 * c[0])) for c in colors]

    # 1) energy graph strip (drawn once, 2*W wide, sits under video+heatmap)
    figp, axp = plt.subplots(figsize=(2 * W / 100.0, 1.9), dpi=100)
    for qi in range(rows * cols):
        axp.plot(times, signal[:, qi], color=colors[qi], lw=0.8)
    axp.axhline(threshold[0], color="k", ls="--", lw=0.8)
    if SHOW_ACTIVATIONS:
        for s, e in _runs(active.any(axis=1)):
            axp.axvspan(times[s], times[e], color="orange", alpha=0.16)
    axp.set_xlim(times[0], times[-1]); axp.set_ylim(0, None); axp.set_yticks([])
    axp.set_xlabel("time (s) - energy per quadrant", fontsize=7); axp.tick_params(labelsize=6)
    figp.subplots_adjust(left=0.01, right=0.999, top=0.97, bottom=0.22)
    figp.canvas.draw()
    strip = np.asarray(figp.canvas.buffer_rgba())[:, :, :3][:, :, ::-1].copy()   # BGR
    bb = axp.get_window_extent(); gx0, gx1 = bb.x0, bb.x1
    plt.close(figp)
    if strip.shape[1] != 2 * W:
        sx = 2 * W / strip.shape[1]
        strip = cv2.resize(strip, (2 * W, int(strip.shape[0] * sx)))
        gx0 *= sx; gx1 *= sx
    SH = strip.shape[0]

    # 2) stream: video | heatmap + grid/active borders, stacked with the cursor graph
    use_ff = shutil.which("ffmpeg") is not None
    proc = writer = None
    tmp = out_path + ".mp4v.mp4"
    for i in range(T):
        small = cv2.cvtColor(frames_rgb[i], cv2.COLOR_RGB2BGR)
        hm = np.clip(std_maps[i] / heat_vmax * 255, 0, 255).astype(np.uint8)
        heat = cv2.applyColorMap(hm, cv2.COLORMAP_INFERNO)
        act_row = active[i] if SHOW_ACTIVATIONS else np.zeros(active.shape[1], dtype=bool)
        draw_grid(small, h, W, rows, cols, bgr, act_row)
        draw_grid(heat, h, W, rows, cols, bgr, act_row)
        top = np.hstack([small, heat])
        cur = int(round(gx0 + (i / max(T - 1, 1)) * (gx1 - gx0)))
        si = strip.copy(); cv2.line(si, (cur, 0), (cur, SH - 1), (30, 30, 30), 1)
        comp = np.ascontiguousarray(np.vstack([top, si]))
        if proc is None and writer is None:
            Hc, Wc = comp.shape[:2]
            if use_ff:
                proc = subprocess.Popen(
                    ["ffmpeg", "-y", "-f", "rawvideo", "-pix_fmt", "bgr24", "-s", f"{Wc}x{Hc}",
                     "-r", f"{fps:.4f}", "-i", "-", "-c:v", "libx264", "-pix_fmt", "yuv420p",
                     "-crf", "30", "-preset", "veryfast", "-loglevel", "error", out_path],
                    stdin=subprocess.PIPE)
            else:
                writer = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*"mp4v"), fps, (Wc, Hc))
        if proc is not None:
            proc.stdin.write(comp.tobytes())
        else:
            writer.write(comp)
    if proc is not None:
        proc.stdin.close(); proc.wait()
    elif writer is not None:
        writer.release(); os.replace(tmp, out_path)
    return out_path

## State and video loading

`VIDEO_PATH` / `fps` are set when you load a video. `STATE` holds the latest detection result
and `PLAYER` is the path of the regenerated clip.

In [ ]:
VIDEO_PATH = None
fps = 10
RESIZE = 320
opts = VideoReadOptions(ignore_edit_list=True)
STATE = {}
PLAYER = str(Path(tempfile.gettempdir()) / "motion_tuning_player.mp4")
SHOW_ACTIVATIONS = True   # draw the active-cell highlight (+ active shading on the energy strip)

In [ ]:
def recompute(responsiveness, sensitivity, grid_quadrants, preview_seconds):
    """Run detection + rebuild visuals for the current preview window; store in STATE."""
    cfg = MotionDetectionOptions.from_gui(
        responsiveness, sensitivity, grid_quadrants,
        duration_s=preview_seconds, resize_width=RESIZE, store=False)
    energies, threshold = motion_detection(VIDEO_PATH, config=cfg, options=opts)

    rows, cols, names, colors = make_grid_visual(cfg.n_quadrants)
    fi = sorted(energies.keys())
    T_full = fi[-1] + 1 if fi else 0
    signal = np.zeros((T_full, cfg.n_quadrants), dtype=float)
    active = np.zeros((T_full, cfg.n_quadrants), dtype=bool)
    for k, entry in energies.items():
        for qi, name in enumerate(names):
            if name in entry:
                signal[k, qi] = entry[name]
                active[k, qi] = True
    times = np.arange(T_full, dtype=float) / fps

    frames_rgb, std_maps = collect_visuals(VIDEO_PATH, cfg, fps, duration_seconds=preview_seconds)
    T = min(T_full, frames_rgb.shape[0])          # guard against decode-count differences
    signal, active, times = signal[:T], active[:T], times[:T]

    STATE.update(dict(signal=signal, active=active, times=times, threshold=threshold,
                      frames_rgb=frames_rgb, std_maps=std_maps, names=names, colors=colors,
                      rows=rows, cols=cols, T=T, cfg=cfg))
    return T


def regenerate():
    """Re-run detection for the current sliders and (re)encode the preview clip."""
    T = recompute(resp.value, sens.value, grid.value, prev.value)
    print(f"recomputed {T} frames @ {fps:.1f} fps; encoding preview clip...")
    path = make_player_from_data(STATE["frames_rgb"], STATE["std_maps"], STATE["active"],
                                 STATE["signal"], STATE["times"], STATE["threshold"],
                                 STATE["rows"], STATE["cols"], STATE["colors"], fps,
                                 out_path=PLAYER)
    out.clear_output(wait=True)
    with out:
        show_video(path)
    print(f"done -> {path}")

## Load a video

Type a path and press **Load video** (or run the cell). The first clip is generated immediately.

In [ ]:
def load_video(path):
    global VIDEO_PATH, fps
    p = Path(str(path).strip())
    if not p.exists():
        print(f"file not found: {p}")
        return
    VIDEO_PATH = str(p)
    props = get_video_properties(VIDEO_PATH)
    fps = float(props.fps or 25.0)
    print(f"loaded: {VIDEO_PATH}\n{props.fps} fps, {props.total_frames} frames, "
          f"{props.width}x{props.height}")
    regenerate()

## Interactive tuning

Drag **Responsiveness / Sensitivity / Grid** and release to regenerate the clip; **Preview sec**
controls how much of the video is analysed (and how long encoding takes). Press play on the
video below to watch it with the overlays.

In [ ]:
path_text = widgets.Text(value="", placeholder="/path/to/video.mp4", description="Video path:")
load_btn = widgets.Button(description="Load video", button_style="primary")

resp = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.5, description="Responsiveness",
                           continuous_update=False)
sens = widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.5, description="Sensitivity",
                           continuous_update=False)
grid = widgets.IntSlider(min=2, max=16, step=1, value=4, description="Grid (cells)",
                         continuous_update=False)
prev = widgets.IntSlider(min=2, max=60, step=1, value=15, description="Preview sec",
                         continuous_update=False)

out = widgets.Output()


def on_load(b):
    load_video(path_text.value)

def on_params(change):
    regenerate()

load_btn.on_click(on_load)
for w in (resp, sens, grid, prev):
    w.observe(on_params, names="value")

display(widgets.HBox([path_text, load_btn]))
display(widgets.HBox([resp, sens, grid, prev]))
display(out)